<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #FFFFFF; max-width: 90%; overflow-x: auto; color: #000000;">

<img src="../../resources/swdb_logo.jpg">

<h1 align="center">Mini-Workshop 1: Clusters from Nothing</h1>
<h3 align="center">Summer Workshop on the Dynamic Brain</h3>
<h4 align="center">Thursday, August 27th, 2026</h4>
<h4 align="center">Day 4</h4>

---

***Authors:** Nick Steinmetz, Carrie Stine*

---

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; max-width: 90%; overflow-x: auto; color: #000000;">

# The Setup: 
A colleague recorded 300 neurons over a 5-second trial and ran a standard
clustering analysis to classify them into functional types. The analysis uses
silhouette scoring to objectively choose the number of clusters, runs k-means,
and displays per-cluster heatmaps and mean response profiles.

Run the cells below to reproduce the analysis, then work through the exercise
to evaluate whether the conclusion holds up.

</div>

In [ ]:
# Setup (imports and plotting defaults)
import os
os.chdir('/code/mini-workshops/nb1')
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.ndimage import gaussian_filter1d

plt.rcParams['font.family']       = 'sans-serif'
plt.rcParams['font.sans-serif']   = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['pdf.fonttype']      = 42
plt.rcParams['ps.fonttype']       = 42
rng = np.random.default_rng(0)   # for cosmetic row shuffling only

HEAT_CMAP = 'inferno'   # perceptually-uniform colormap for heatmaps

def cluster_palette(n):
    """n distinct qualitative colors, one per cluster."""
    return plt.cm.tab10(np.arange(n) % 10)

def cluster_label(c, k):
    """Human-readable cluster name. For k=3 use early/middle/late."""
    if k == 3:
        return f'cluster {c} ({["early", "middle", "late"][c]})'
    return f'cluster {c}'


In [ ]:
# Load the dataset: a (neurons x time) matrix of activity.
d = np.load('data/activity.npz')
activity = d['activity']      # shape (n_neurons, n_time)
time     = d['time']          # seconds
n_neurons, n_time = activity.shape
print(f'{n_neurons} neurons x {n_time} time bins ({time[0]:.2f}-{time[-1]:.2f} s)')


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; max-width: 90%; overflow-x: auto; color: #000000;">

## The raw data

Each row is a neuron, each column a time bin. This is the matrix everything
below is computed from.


</div>


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(activity, aspect='auto', cmap=HEAT_CMAP,
               vmin=np.percentile(activity, 2), vmax=np.percentile(activity, 99),
               extent=[time[0], time[-1], n_neurons, 0])
ax.set_xlabel('time (s)'); ax.set_ylabel('neuron')
ax.set_title('Activity matrix (unsorted)')
plt.colorbar(im, ax=ax, label='activity (a.u.)')
plt.show()


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; max-width: 90%; overflow-x: auto; color: #000000;">

## Step 1: How many clusters?

Rather than picking the number of clusters by hand, the standard approach is
**silhouette analysis** - scan a range of k values, score each clustering by
how tightly points sit within their own cluster relative to the nearest other
cluster, and keep the k with the best score.

</div>


In [ ]:
# Scan candidate cluster counts k=2 through 8.
# For each k, run k-means and compute the silhouette score.
# Higher silhouette = tighter, more separated clusters.
candidate_ks = range(2, 9)
sil = {kk: silhouette_score(
           activity,
           KMeans(n_clusters=kk, n_init=10, random_state=0).fit_predict(activity))
       for kk in candidate_ks}
best_k = max(sil, key=sil.get)
print(f'Silhouette analysis selects k = {best_k} clusters.')
print(f'Silhouette scores: {", ".join(f"k={k}: {v:.3f}" for k, v in sil.items())}')


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; max-width: 90%; overflow-x: auto; color: #000000;">

## Step 2: Cluster the neurons

Silhouette analysis picked three clusters, so we treat each neuron's time
course as a feature vector and run k-means with `k = 3`.

</div>


In [ ]:
k = 3
km     = KMeans(n_clusters=k, n_init=10, random_state=0)
labels = km.fit_predict(activity)   # assign each neuron to a cluster

# Relabel clusters 0/1/2 = early/middle/late by their mean-trace peak time.
# This is cosmetic only - it does not change the clustering.
peak_of_mean = np.array([activity[labels == c].mean(0).argmax() for c in range(k)])
order        = np.argsort(peak_of_mean)
relabel      = np.zeros(k, dtype=int)
relabel[order] = np.arange(k)
labels = relabel[labels]

for c in range(k):
    print(f'{cluster_label(c, k)}: {(labels == c).sum()} neurons')


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; max-width: 90%; overflow-x: auto; color: #000000;">

## Step 3: Visualize the clusters

Two views:
1. **Heatmaps** of the neurons in each cluster (rows in arbitrary order).
2. **Mean response** of each cluster (mean +/- SEM across neurons).

</div>


In [ ]:
colors = cluster_palette(k)
fig    = plt.figure(figsize=(11, 1.6 * k + 0.6))
gs     = fig.add_gridspec(k, 2, width_ratios=[1, 1.1], hspace=0.7, wspace=0.3)

vmin = np.percentile(activity, 2)
vmax = np.percentile(activity, 99)

# Left column: one heatmap per cluster (rows shuffled within cluster for display)
heat_axes = []
for c in range(k):
    ax  = fig.add_subplot(gs[c, 0])
    idx = rng.permutation(np.where(labels == c)[0])
    im  = ax.imshow(activity[idx], aspect='auto', cmap=HEAT_CMAP,
                    vmin=vmin, vmax=vmax,
                    extent=[time[0], time[-1], 0, len(idx)])
    ax.set_title(cluster_label(c, k), color=colors[c], fontweight='bold')
    ax.set_ylabel('neuron')
    if c == k - 1:
        ax.set_xlabel('time (s)')
    heat_axes.append(ax)
plt.colorbar(im, ax=heat_axes, fraction=0.046, pad=0.02).set_label('activity (a.u.)')

# Right column: mean +/- SEM trace per cluster
ax = fig.add_subplot(gs[:, 1])
for c in range(k):
    m   = activity[labels == c].mean(0)
    sem = activity[labels == c].std(0) / np.sqrt((labels == c).sum())
    ax.plot(time, m, color=colors[c], lw=2, label=cluster_label(c, k))
    ax.fill_between(time, m - sem, m + sem, color=colors[c], alpha=0.3)
ax.set_xlabel('time (s)'); ax.set_ylabel('activity (a.u.)')
ax.set_title('Mean response by cluster'); ax.legend(frameon=False)

fig.suptitle('Three temporal response classes?', fontsize=14, fontweight='bold')
plt.show()


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; max-width: 90%; overflow-x: auto; color: #000000;">

The silhouette analysis selected k=3. The three clusters show clearly separated
mean traces and each cluster's heatmap appears internally coherent. A colleague
concludes:

> *"The population contains three distinct functional classes of neurons -
> **early**, **middle**, and **late** responders - each with its own reliable
> temporal response profile."*

**The claim is false.** Work through the exercise below to find out why.

</div>


<div style="border-left: 3px solid #07bc0a; padding: 1px; padding-left: 10px; background: #DFF0D8; max-width: 90%; overflow-x: auto; color: #000000;">

### Exercise: Evaluate the cluster analysis

Before accepting the conclusion of three response classes, work through the
following questions. The goal is to identify what the analysis assumes and
what checks it omits.

<ol>

<li><strong>What does k-means guarantee?</strong> Given that you ask for k=3 groups,
will k-means ever return fewer - for example, if the data show no natural
separation? What does this mean for interpreting the output?
<details>
<summary>Hint</summary>

*k-means is required to assign every data point to exactly one of k groups.
It will always return k non-empty clusters regardless of whether any real
grouping exists. Try changing <code>k = 3</code> to <code>k = 4</code> or <code>k = 5</code> in the
clustering cell and re-run - what do you get?*

</details>
</li>

<br>

<li><strong>How large is the silhouette score?</strong> Look at the values printed above.
As a rough rule of thumb, a silhouette below ~0.25 indicates no substantial
cluster structure. What does the peak value here suggest?
<details>
<summary>Hint</summary>

*A silhouette score always names a "best k" - but the value tells you how
real the structure is. A low peak value at k=3 means k=3 is the best of a
set of poor options, not evidence that 3 clusters genuinely exist.*

</details>
</li>

<br>

<li><strong>Design a check.</strong> If there truly were three discrete response classes,
each neuron would belong to one class and its peak activity time would fall
within a narrow range for that class. What would the <em>distribution</em> of
peak times look like across all neurons for three real classes? For a smooth
continuum?

Estimate each neuron's peak time (lightly smooth its trace and take the
argmax), then plot the distribution. Does it look trimodal?
<details>
<summary>Hint</summary>

*Use <code>gaussian_filter1d(activity, sigma=2.0, axis=1)</code> to smooth, then
<code>time[smoothed.argmax(1)]</code> to get each neuron's peak time in seconds.
Plot as a histogram.*

</details>
</li>

<br>

<li><strong>Visualize without presupposing groups.</strong> The per-cluster heatmaps split
neurons into three panels, which invites the eye to see three blocks. Sort
<em>all</em> neurons together by peak time and display them in a single heatmap.
What does the full population look like?
<details>
<summary>Hint</summary>

*Sort the rows of <code>activity</code> by <code>np.argsort(peak_time_est)</code> and display
with <code>imshow</code>. Optionally mark where k-means draws its boundaries.*

</details>
</li>

</ol>

</div>


## Solution


In [ ]:
# ── Evidence 1: Distribution of peak times ────────────────────────────────
# Lightly smooth each neuron's trace, then take the time of its maximum
# as an estimate of that neuron's peak time.



In [ ]:
# ── Evidence 2: Sort all neurons by peak time in one heatmap ──────────────
# The per-cluster heatmaps used three separate panels and arbitrary row order,
# which invites the eye to see three distinct blocks. Putting all neurons in
# one heatmap, sorted by peak time, may help reveal the truth.



In [1]:
# ── Evidence 3: Does PCA show 3 distinct clusters? ──────────
# If three discrete clusters existed, we would see three separated clouds in
# PC space. 


In [ ]:
# ── Evidence 4: Strength of the silhouette peak at k=3 ───────
# Silhouette always names a "best k". The decisive check: compare the real
# data against no-cluster surrogates - data we KNOW has no groups.
# If the peak at k=3 were evidence of clusters, these surrogates should not
# reproduce it.




<div style="border-left: 3px solid #f0b429; padding: 1px; padding-left: 10px; background: #FFF9C4;  max-width: 90%; overflow-x: auto; color: #000000;">

## Key takeaway

<details>
<summary><b>reveal after completing the exercises!</b></summary>
There are no clusters. Every neuron has a single smooth bump peaking at a
randomly chosen, continuous time. K-means with k=3 simply sliced that continuum
into three contiguous bins and called them classes.

- **k-means always returns k clusters.** Asking for 3 groups guarantees 3
  groups, whether or not any exist. The output is not evidence that discrete
  classes are present.
- **A continuum will be sliced into contiguous bins.** When a single latent
  variable (here, peak time) varies smoothly, clustering carves it into
  arbitrary "types" whose boundaries are meaningless.
- **Averaging within groups hides the within-group distribution.** The tidy
  mean +/- SEM traces conceal that peak times vary continuously across the
  whole trial; the SEM is small only because n is large, not because the group
  is homogeneous.
- **A "best k" is not evidence of clusters.** Silhouette (or gap statistic,
  BIC, etc.) selects the best k *assuming you cluster at all*. It happily peaks
  on a continuum — the peak here is reproduced by data we know has no groups.
  Always read the *value*, and compare it against a matched no-cluster null.
- **Averaging within groups hides the within-group distribution.** The tidy
  mean +/- SEM traces conceal that peak times vary continuously across the
  whole trial; the SEM is small only because n is large, not because the group
  is homogeneous.

**How to do it right:** before interpreting clusters as types, ask whether a
continuous model explains the data at least as well, compare any cluster-validity
metric against a matched null (not just its own maximum), and visualize the data
in a way that does not presuppose the number of groups.

</details>

<br>

</div>